In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

print("Data folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

Data folder: /Users/melqueso/Documents/QSS45-final-project/data
Output folder: /Users/melqueso/Documents/QSS45-final-project/output


In [2]:
water = pd.read_csv(DATA_DIR / "water.csv")
rents = pd.read_csv(DATA_DIR / "resource_rents.csv")
gdp = pd.read_csv(DATA_DIR / "gdp.csv")
electricity = pd.read_csv(DATA_DIR / "electricity.csv")

print("Water:", water.shape)
print("Rents:", rents.shape)
print("GDP:", gdp.shape)
print("Electricity:", electricity.shape)

Water: (6061, 4)
Rents: (9911, 4)
GDP: (7445, 5)
Electricity: (7140, 4)


In [3]:
def merge_check(left, right, name):
    print(f"\nBefore {name}:")
    print("Left rows:", len(left))
    print("Right rows:", len(right))

    merged = left.merge(
        right,
        on=["Entity", "Code", "Year"],
        how="inner"
    )

    print(f"After {name}:")
    print("Merged rows:", len(merged))

    return merged

In [4]:
water = water.rename(columns={
    "Share of the population using at least a basic drinking water source": "water_access"
})

rents = rents.rename(columns={
    "Total natural resources rents (% of GDP)": "resource_rents"
})

gdp = gdp.rename(columns={
    "GDP per capita": "gdp_per_capita",
    "World region according to OWID": "region"
})

electricity = electricity.rename(columns={
    "Share of the population with access to electricity": "electricity_access"
})

print(water.columns.tolist())
print(rents.columns.tolist())
print(gdp.columns.tolist())
print(electricity.columns.tolist())

['Entity', 'Code', 'Year', 'water_access']
['Entity', 'Code', 'Year', 'resource_rents']
['Entity', 'Code', 'Year', 'gdp_per_capita', 'region']
['Entity', 'Code', 'Year', 'electricity_access']


In [5]:
gdp_africa = gdp[
    gdp["region"] == "Africa"
].copy()

print("African GDP rows:", len(gdp_africa))
print("African countries:", gdp_africa["Entity"].nunique())

display(gdp_africa.head())

African GDP rows: 1848
African countries: 52


,Entity,Code,Year,gdp_per_capita,region
61,Algeria,DZA,1990,11728.546,Africa
62,Algeria,DZA,1991,11314.864,Africa
63,Algeria,DZA,1992,11241.415,Africa
64,Algeria,DZA,1993,10743.706,Africa
65,Algeria,DZA,1994,10414.035,Africa


In [7]:
merged = merge_check(
    water,
    rents,
    "water + resource rents"
)


Before water + resource rents:
Left rows: 6061
Right rows: 9911
After water + resource rents:
Merged rows: 4512


In [8]:
merged = merge_check(
    merged,
    gdp_africa[["Entity", "Code", "Year", "gdp_per_capita", "region"]],
    "GDP"
)


Before GDP:
Left rows: 4512
Right rows: 1848
After GDP:
Merged rows: 1101


In [9]:
merged = merge_check(
    merged,
    electricity,
    "electricity"
)


Before electricity:
Left rows: 1101
Right rows: 7140
After electricity:
Merged rows: 1092


In [10]:
merged = merged[
    (merged["Year"] >= 2000) &
    (merged["Year"] <= 2021)
].copy()

print("Rows from 2000-2021:", len(merged))
print("Countries:", merged["Entity"].nunique())
print("Year range:", merged["Year"].min(), "-", merged["Year"].max())

Rows from 2000-2021: 1092
Countries: 52
Year range: 2000 - 2021


In [11]:
variables = [
    "water_access",
    "resource_rents",
    "gdp_per_capita",
    "electricity_access"
]

print("Missing values before cleaning:")
print(merged[variables].isna().sum())

Missing values before cleaning:
water_access          0
resource_rents        0
gdp_per_capita        0
electricity_access    0
dtype: int64


In [12]:
analysis = merged.dropna(
    subset=variables
).copy()

analysis = analysis[
    analysis["gdp_per_capita"] > 0
].copy()

print("Rows after removing missing values:", len(analysis))
print("Countries after cleaning:", analysis["Entity"].nunique())

Rows after removing missing values: 1092
Countries after cleaning: 52


In [13]:
analysis["log_gdp_pc"] = np.log(
    analysis["gdp_per_capita"]
)

display(
    analysis[
        [
            "Entity",
            "Code",
            "Year",
            "water_access",
            "resource_rents",
            "gdp_per_capita",
            "log_gdp_pc",
            "electricity_access"
        ]
    ].head()
)

,Entity,Code,Year,water_access,resource_rents,gdp_per_capita,log_gdp_pc,electricity_access
0,Algeria,DZA,2000,88.993805,26.686897,11558.221,9.355152,98.6
1,Algeria,DZA,2001,89.169800,23.630941,11742.595,9.370978,98.6
2,Algeria,DZA,2002,89.343640,23.470938,12213.126,9.410267,98.6
3,Algeria,DZA,2003,89.515110,24.452322,12835.182,9.459945,98.6
4,Algeria,DZA,2004,89.684220,26.157312,13226.765,9.489998,98.6


In [14]:
analysis = analysis.sort_values(
    ["Entity", "Year"]
).reset_index(drop=True)

print("FINAL DATASET")
print("----------------")
print("Observations:", len(analysis))
print("Countries:", analysis["Entity"].nunique())
print("Years:", analysis["Year"].min(), "-", analysis["Year"].max())

print("\nSummary statistics:")
display(
    analysis[
        [
            "water_access",
            "resource_rents",
            "gdp_per_capita",
            "log_gdp_pc",
            "electricity_access"
        ]
    ].describe()
)

FINAL DATASET
----------------
Observations: 1092
Countries: 52
Years: 2000 - 2021

Summary statistics:


,water_access,resource_rents,gdp_per_capita,log_gdp_pc,electricity_access
count,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000
mean,65.630329,11.358810,6271.091146,8.333037,44.426923
std,17.946497,11.260904,6347.715791,0.889488,29.616142
min,18.759062,0.002360,711.976400,6.568045,1.300000
25%,52.251223,3.646283,2195.359100,7.694101,17.925000
50%,64.525797,7.586374,3588.570800,8.185509,39.850000
75%,79.479637,14.933098,8207.746750,9.012832,65.000000
max,99.853745,88.592350,39994.617000,10.596500,100.000000


In [15]:
burkina = analysis[
    analysis["Entity"] == "Burkina Faso"
]

print("Burkina Faso observations:", len(burkina))

display(
    burkina[
        [
            "Year",
            "water_access",
            "resource_rents",
            "gdp_per_capita",
            "electricity_access"
        ]
    ].head()
)

Burkina Faso observations: 22


,Year,water_access,resource_rents,gdp_per_capita,electricity_access
88,2000,55.639090,5.334609,1428.5740,9.1
89,2001,55.832905,4.767833,1477.2849,9.5
90,2002,55.611430,5.728161,1494.6780,9.9
91,2003,55.390312,6.323635,1561.8331,11.4
92,2004,55.171288,5.437260,1581.2562,10.7


In [16]:
analysis.to_csv(
    DATA_DIR / "analysis_data.csv",
    index=False
)

print("Saved:", DATA_DIR / "analysis_data.csv")

Saved: /Users/melqueso/Documents/QSS45-final-project/data/analysis_data.csv


In [17]:
merged = merge_check(
    merged,
    electricity,
    "electricity"
)


Before electricity:
Left rows: 1092
Right rows: 7140
After electricity:
Merged rows: 1092


In [23]:
merged = merged[
    (merged["Year"] >= 2000) &
    (merged["Year"] <= 2021)
].copy()

print("Rows from 2000-2021:", len(merged))
print("Countries:", merged["Entity"].nunique())
print("Year range:", merged["Year"].min(), "-", merged["Year"].max())

Rows from 2000-2021: 1092
Countries: 52
Year range: 2000 - 2021


In [26]:
# Combine the duplicate electricity columns into one
merged["electricity_access"] = merged["electricity_access_x"].combine_first(
    merged["electricity_access_y"]
)

# Remove the duplicate columns
merged = merged.drop(
    columns=["electricity_access_x", "electricity_access_y"]
)

print(merged.columns.tolist())

['Entity', 'Code', 'Year', 'water_access', 'resource_rents', 'gdp_per_capita', 'region', 'electricity_access']


In [27]:
variables = [
    "water_access",
    "resource_rents",
    "gdp_per_capita",
    "electricity_access"
]

print("Missing values before cleaning:")
print(merged[variables].isna().sum())

Missing values before cleaning:
water_access          0
resource_rents        0
gdp_per_capita        0
electricity_access    0
dtype: int64


In [28]:
analysis = merged.copy()

analysis = analysis[
    analysis["gdp_per_capita"] > 0
].copy()

print("Rows after cleaning:", len(analysis))
print("Countries after cleaning:", analysis["Entity"].nunique())

Rows after cleaning: 1092
Countries after cleaning: 52


In [29]:
analysis["log_gdp_pc"] = np.log(
    analysis["gdp_per_capita"]
)

In [30]:
analysis = analysis.sort_values(
    ["Entity", "Year"]
).reset_index(drop=True)

print("FINAL DATASET")
print("----------------")
print("Observations:", len(analysis))
print("Countries:", analysis["Entity"].nunique())
print("Years:", analysis["Year"].min(), "-", analysis["Year"].max())

display(
    analysis[
        [
            "water_access",
            "resource_rents",
            "gdp_per_capita",
            "log_gdp_pc",
            "electricity_access"
        ]
    ].describe()
)

FINAL DATASET
----------------
Observations: 1092
Countries: 52
Years: 2000 - 2021


,water_access,resource_rents,gdp_per_capita,log_gdp_pc,electricity_access
count,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000
mean,65.630329,11.358810,6271.091146,8.333037,44.426923
std,17.946497,11.260904,6347.715791,0.889488,29.616142
min,18.759062,0.002360,711.976400,6.568045,1.300000
25%,52.251223,3.646283,2195.359100,7.694101,17.925000
50%,64.525797,7.586374,3588.570800,8.185509,39.850000
75%,79.479637,14.933098,8207.746750,9.012832,65.000000
max,99.853745,88.592350,39994.617000,10.596500,100.000000


In [31]:
analysis.to_csv(
    DATA_DIR / "analysis_data.csv",
    index=False
)

print("Saved:", DATA_DIR / "analysis_data.csv")

Saved: /Users/melqueso/Documents/QSS45-final-project/data/analysis_data.csv
